In [2]:
import time
import random
import re
import warnings

import numpy as np
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

warnings.filterwarnings("ignore")

# -----------------------------
# Configuration
# -----------------------------
INPUT_FILE = "Airbnb_Combined_Data.csv"
OUTPUT_FILE = "Airbnb_Combined_Enriched.csv"

HEADLESS = True        # Change to False if you want to watch the browser
WAIT_TIME = 10
MAX_RESULTS_PER_ROW = 5

In [3]:
# -----------------------------
# Read Dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE)

# Create row_id (used later if needed)
df = df.reset_index(drop=True)
df["row_id"] = df.index

print(f"Dataset Shape: {df.shape}")

# Cities to scrape
cities = sorted(df["city"].unique())

print(f"\nCities ({len(cities)}):")
print(cities)

# We'll scrape 50 random listings per city
TARGET_PER_CITY = 50

# Store scraped data
all_data = []

print(f"\nTarget Listings = {TARGET_PER_CITY} per city")
print(f"Expected Total = {TARGET_PER_CITY * len(cities)}")

Dataset Shape: (51707, 23)

Cities (10):
['Amsterdam', 'Athens', 'Barcelona', 'Berlin', 'Budapest', 'Lisbon', 'London', 'Paris', 'Rome', 'Vienna']

Target Listings = 50 per city
Expected Total = 500


In [4]:
# =====================================
# Selenium Driver
# =====================================

chrome_options = Options()

if HEADLESS:
    chrome_options.add_argument("--headless=new")

chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Anti detection
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option(
    "excludeSwitches",
    ["enable-automation"]
)
chrome_options.add_experimental_option(
    "useAutomationExtension",
    False
)

driver = webdriver.Chrome(options=chrome_options)

driver.execute_script("""
Object.defineProperty(
    navigator,
    'webdriver',
    {get: () => undefined}
);
""")

wait = WebDriverWait(driver, WAIT_TIME)

print("Chrome Started Successfully!")

Chrome Started Successfully!


In [6]:
# =====================================
# Sample the Dataset
# =====================================

TARGET_PER_CITY = 15

sample_df = (
    df.groupby("city", group_keys=False)
      .apply(lambda x: x.sample(
          min(TARGET_PER_CITY, len(x)),
          random_state=42
      ))
      .reset_index(drop=True)
)

sample_df["row_id"] = sample_df.index

print(sample_df.shape)

sample_df.head()

(150, 23)


,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type,row_id
0,261,550.465165,Entire home/apt,False,False,2.0,False,1,0,10.0,...,1.703751,203.098080,10.753113,247.152944,21.545804,4.85826,52.36067,Amsterdam,weekends,0
1,422,409.392356,Entire home/apt,False,False,2.0,False,0,0,9.0,...,1.481714,182.351858,9.655652,223.585417,15.579754,4.85470,52.36110,Amsterdam,weekdays,1
2,538,1892.531577,Entire home/apt,False,False,6.0,False,0,0,10.0,...,0.154896,407.104748,21.556467,487.487990,33.968865,4.89118,52.35495,Amsterdam,weekdays,2
3,29,270.428608,Private room,False,True,4.0,True,0,0,10.0,...,1.094837,91.666212,4.853787,114.674343,7.990673,4.82174,52.37781,Amsterdam,weekdays,3
4,825,952.827315,Entire home/apt,False,False,4.0,True,0,0,10.0,...,0.194053,157.299088,8.328266,197.942322,17.255819,4.89522,52.34115,Amsterdam,weekends,4


In [11]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# =====================================
# Configuration
# =====================================

LISTINGS_PER_CITY = 5
HEADLESS_MODE = False  # Set to False to see browser (helps debug)
MAX_RETRIES = 2

# =====================================
# Utility Functions
# =====================================

def random_sleep(min_time=1, max_time=3):
    """Sleep for a random amount of time."""
    time.sleep(random.uniform(min_time, max_time))

def extract_price(driver):
    """Extract price from the listing page - much more robust."""
    try:
        # First, try the search results page price (faster)
        # Then try various listing page selectors
        page_text = driver.page_source
        
        # Look for price patterns in the page text
        import re
        price_matches = re.findall(r'[\$€£]\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?', page_text)
        if price_matches:
            return price_matches[0]
        
        # Try various selectors
        selectors = [
            "span._1y74zjx",
            "span[data-testid='price-per-night']",
            "span._tyxjp1",
            "div._1jo4hgw span",
            "span[data-testid='price-amount']",
            "div[data-testid='book-it-default'] span",
            "div.g1tup83b span"
        ]
        
        for selector in selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    text = el.text.strip()
                    if text and (text.startswith('$') or text.startswith('€') or text.startswith('£') or any(c.isdigit() for c in text)):
                        return text
            except:
                continue
                
    except Exception as e:
        print(f"      Price extract error: {str(e)[:40]}")
    
    return "N/A"

def extract_superhost(driver):
    """Extract superhost status from the listing page."""
    try:
        page_text = driver.page_source.lower()
        if "superhost" in page_text:
            return True
        
        # Also check elements
        selectors = [
            "div[data-testid='SuperhostBadge']",
            "span[data-testid='SuperhostBadge']",
            "[class*='superhost']"
        ]
        
        for selector in selectors:
            try:
                driver.find_element(By.CSS_SELECTOR, selector)
                return True
            except:
                continue
    except:
        pass
    
    return False

def extract_rating_reviews(driver):
    """Extract rating, reviews, and guest favorite status from the listing page."""
    rating = "N/A"
    reviews = "N/A"
    guest_favorite = False
    
    try:
        page_text = driver.page_source
        
        # Look for rating pattern (e.g., "4.92" or "4,92")
        import re
        rating_matches = re.findall(r'\b\d[.,]\d{1,2}\b', page_text)
        for match in rating_matches:
            # Check if it's a reasonable rating (0-5)
            try:
                val = float(match.replace(',', '.'))
                if 0 <= val <= 5:
                    rating = match
                    break
            except:
                continue
        
        # Look for review count pattern
        review_matches = re.findall(r'(\d{1,3}(?:,\d{3})*)\s*reviews?', page_text, re.IGNORECASE)
        if review_matches:
            reviews = review_matches[0]
        
        # Check for Guest Favorite
        if "guest favorite" in page_text.lower():
            guest_favorite = True
        
        # Also try CSS selectors as backup
        if rating == "N/A":
            rating_selectors = [
                "span._12si43g",
                "div[data-testid='reviews'] span",
                "span.r1lutz1s",
                "div.a8jt5op"
            ]
            for selector in rating_selectors:
                try:
                    el = driver.find_element(By.CSS_SELECTOR, selector)
                    text = el.text.strip()
                    if text and len(text) <= 4 and ('.' in text or ',' in text):
                        rating = text
                        break
                except:
                    continue
        
        if reviews == "N/A":
            review_selectors = [
                "span._1jl0641",
                "span[data-testid='reviews-count']",
                "div[data-testid='reviews'] span:nth-child(2)"
            ]
            for selector in review_selectors:
                try:
                    el = driver.find_element(By.CSS_SELECTOR, selector)
                    text = el.text.strip()
                    if text and any(c.isdigit() for c in text):
                        reviews = text
                        break
                except:
                    continue
        
        if not guest_favorite:
            try:
                driver.find_element(By.CSS_SELECTOR, "div[data-testid='GuestFavoriteBadge']")
                guest_favorite = True
            except:
                pass
                
    except Exception as e:
        print(f"      Rating/review extract error: {str(e)[:40]}")
    
    return rating, reviews, guest_favorite

def init_driver():
    """Initialize Chrome driver (headless or visible)."""
    chrome_options = Options()
    if HEADLESS_MODE:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    
    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 10)  # Shorter timeout
    return driver, wait

# =====================================
# Scrape Listings
# =====================================

def scrape_listing(driver, wait, listing_url, row_data):
    """Scrape a single listing page with better error handling."""
    main_window = driver.current_window_handle
    
    for attempt in range(MAX_RETRIES):
        try:
            driver.set_page_load_timeout(15)  # 15 second timeout
            driver.execute_script(f"window.open('{listing_url}','_blank');")
            driver.switch_to.window(driver.window_handles[-1])
            
            # Wait for page to load with shorter timeout
            try:
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
            except:
                pass  # Continue even if h1 not found immediately
            
            random_sleep(0.5, 1)
            
            try:
                name = driver.find_element(By.TAG_NAME, "h1").text
            except:
                name = "N/A"
            
            price = extract_price(driver)
            superhost = extract_superhost(driver)
            rating, reviews, guest_favorite = extract_rating_reviews(driver)
            
            result = {
                "row_id": row_data["row_id"],
                "Name": name,
                "city": row_data["city"],
                "Price": price,
                "Rating": rating,
                "Reviews": reviews,
                "Superhost": superhost,
                "Guest_Favorite": guest_favorite,
                "URL": listing_url
            }
            
            # Close tab and switch back
            driver.close()
            driver.switch_to.window(main_window)
            return result
            
        except Exception as e:
            print(f"    Attempt {attempt+1} failed: {str(e)[:50]}...")
            # Clean up if tab is still open
            try:
                if len(driver.window_handles) > 1:
                    driver.close()
                    driver.switch_to.window(main_window)
            except:
                pass
            if attempt < MAX_RETRIES - 1:
                random_sleep(1, 2)
    
    return None

def scrape_city(driver, wait, row, listings_needed):
    """Scrape listings for a single city - get data from search results first!"""
    city_name = str(row["city"])  # Keep original city name for the data!
    city = city_name.replace(" ", "-")
    
    print(f"Scraping {listings_needed} listings for {city_name}...")
    
    city_listings = []
    page = 0
    items_offset = 0
    max_pages = 10  # More pages to get enough listings
    consecutive_failures = 0
    max_consecutive_failures = 3
    
    while len(city_listings) < listings_needed and page < max_pages:
        if consecutive_failures >= max_consecutive_failures:
            print(f"  Too many failures, stopping at {len(city_listings)} listings")
            break
            
        url = (
            f"https://www.airbnb.com/s/{city}/homes"
            f"?items_offset={items_offset}"
        )
        
        try:
            driver.set_page_load_timeout(30)
            driver.get(url)
            random_sleep(3, 5)  # Give page time to load properly
            
            try:
                cards = wait.until(
                    EC.presence_of_all_elements_located(
                        (By.CSS_SELECTOR, "div[data-testid='card-container']")
                    )
                )
                consecutive_failures = 0
                print(f"  Found {len(cards)} listings on page {page+1}")
            except:
                consecutive_failures += 1
                print(f"  No cards found (attempt {consecutive_failures}/{max_consecutive_failures})")
                page += 1
                items_offset = page * 18
                continue
            
            if not cards:
                break
            
            for i, card in enumerate(cards):
                if len(city_listings) >= listings_needed:
                    break
                
                try:
                    # Extract what we can from the search result first!
                    # Only print debug for first 2 cards to avoid clutter
                    debug = (i < 2 and page == 0)
                    listing_data = extract_from_search_card(card, row, city_name, debug_print=debug)
                    
                    if listing_data:
                        city_listings.append(listing_data)
                        print(f"  [{len(city_listings)}/{listings_needed}] Scraped: {listing_data['Name'][:35]}...")
                        print(f"      Price: {listing_data['Price']} | Rating: {listing_data['Rating']} | Reviews: {listing_data['Reviews']}")
                    else:
                        print(f"  [{len(city_listings)+1}/{listings_needed}] Skipped (no data)")
                    
                except Exception as e:
                    print(f"  Error with listing {i+1}: {str(e)[:40]}...")
                    continue
            
            if len(cards) < 18:
                print(f"  End of results for {city_name}")
                break
            
            page += 1
            items_offset = page * 18
            
        except Exception as e:
            consecutive_failures += 1
            print(f"Error on page {page}: {str(e)[:50]}...")
            page += 1
            items_offset = page * 18
            continue
    
    return city_listings


def extract_from_search_card(card, row, city_name, debug_print=True):
    """Extract listing data directly from the search result card (FAST!)."""
    import re
    
    try:
        # Get URL
        link_element = card.find_element(By.TAG_NAME, "a")
        listing_url = link_element.get_attribute("href")
        
        # Get ALL text from the card and analyze it
        card_text = card.text
        card_html = card.get_attribute("innerHTML")
        
        # Debug: print what we found
        if debug_print:
            print(f"\n[DEBUG] Card text preview: {card_text[:200]}...")
        
        # Extract name
        name = "N/A"
        lines = card_text.split('\n')
        
        # First non-empty line that's not "Guest favorite" is usually the name
        for line in lines:
            stripped = line.strip()
            if stripped and len(stripped) > 3 and stripped.lower() != "guest favorite":
                name = stripped
                break
        
        # Extract price - we might not get this from the card, let's try anyway
        price = "N/A"
        # Look for price in HTML too
        if card_html:
            price_patterns = [
                r'[\$€£]\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?',
                r'\d{1,3}(?:,\d{3})*(?:\.\d{2})?\s?(?:\$|€|£)'
            ]
            for pattern in price_patterns:
                matches = re.findall(pattern, card_html)
                if matches:
                    price = matches[0]
                    break
        
        # Extract rating - look for "X.XX out of 5" pattern
        rating = "N/A"
        rating_match = re.search(r'(\d[.,]\d{1,2})\s+out of 5', card_text, re.IGNORECASE)
        if rating_match:
            rating = rating_match.group(1)
        else:
            # Look for "X.XX (XXX)" format
            rating_match2 = re.search(r'(\d[.,]\d{1,2})\s*\((\d+)\)', card_text)
            if rating_match2:
                rating = rating_match2.group(1)
        
        # Extract reviews - look for "XXX reviews" pattern
        reviews = "N/A"
        rev_match = re.search(r'(\d{1,3}(?:,\d{3})*)\s*reviews?', card_text, re.IGNORECASE)
        if rev_match:
            reviews = rev_match.group(1)
        else:
            # Look for (XXX) format
            rev_match2 = re.search(r'\((\d{1,3}(?:,\d{3})*)\)', card_text)
            if rev_match2:
                reviews = rev_match2.group(1)
        
        # Superhost detection
        superhost = "superhost" in card_text.lower()
        
        guest_favorite = "guest favorite" in card_text.lower()
        
        return {
            "row_id": row["row_id"],
            "Name": name,
            "city": city_name,
            "Price": price,
            "Rating": rating,
            "Reviews": reviews,
            "Superhost": superhost,
            "Guest_Favorite": guest_favorite,
            "URL": listing_url
        }
        
    except Exception as e:
        print(f"      Card extract error: {str(e)}")
        return None

# =====================================
# Main Execution
# =====================================

def main():
    # Initialize driver
    print("Starting scraper (headless mode)...")
    driver, wait = init_driver()
    
    # Load your actual data
    data_path = r"c:\Users\manar\Downloads\AI COURSE\Machine Learning\Sprints\1st Project\airbnb-europe-dwh\5th Other scripts\Airbnb_Combined_Data.csv"
    
    try:
        original_df = pd.read_csv(data_path)
        # Rename the first column to row_id
        original_df = original_df.rename(columns={"Unnamed: 0": "row_id"})
        print(f"Loaded data with {len(original_df)} rows")
        
        # For each unique city, create a representative row with common filters
        unique_cities = original_df["city"].unique()
        print(f"Found {len(unique_cities)} unique cities to scrape: {', '.join(unique_cities)}")
        
    except FileNotFoundError:
        print(f"Data file not found at: {data_path}")
        return
    
    scraped_data = []
    
    try:
        for city in unique_cities:
            # Get a representative row for this city
            city_row = original_df[original_df["city"] == city].iloc[0].copy()
            city_listings = scrape_city(driver, wait, city_row, LISTINGS_PER_CITY)
            scraped_data.extend(city_listings)
            print(f"\nCompleted {city}: {len(city_listings)} listings scraped\n")
    
    finally:
        # Save scraped data
        scraped_df = pd.DataFrame(scraped_data)
        scraped_df.to_csv("airbnb_scraped_data.csv", index=False)
        print(f"\nScraping complete! Total listings scraped: {len(scraped_data)}")
        print("Scraped data saved to airbnb_scraped_data.csv")
        
        # Also save a copy of the original data for reference
        original_df.to_csv("airbnb_original_data.csv", index=False)
        print("Original data saved to airbnb_original_data.csv")
        
        print("\n✅ Done! You now have:")
        print("   - airbnb_scraped_data.csv (50 listings per city, newly scraped)")
        print("   - airbnb_original_data.csv (your original dataset)")
        
        driver.quit()

if __name__ == "__main__":
    main()


Starting scraper (headless mode)...
Loaded data with 51707 rows
Found 10 unique cities to scrape: Amsterdam, Athens, Barcelona, Berlin, Budapest, Lisbon, London, Paris, Rome, Vienna
Scraping 5 listings for Amsterdam...
  Found 28 listings on page 1

[DEBUG] Card text preview: Guest favorite
Guest favorite
Shared room in Amsterdam
The Elephant Hostel - Mixed Dorm 24 Beds
1 bed
1 bed
Jul 8 – 13
Jul 8 – 13
ج.م 13,969
Show price breakdown
 for 5 nights
ج.م 13,969 for 5 nights
...
  [1/5] Scraped: Shared room in Amsterdam...
      Price: N/A | Rating: 4.83 | Reviews: 262

[DEBUG] Card text preview: Superhost
Superhost
Apartment in Amsterdam
2-Bedroom Apartment - ID Aparthotel
2 bedrooms
2 bedrooms
,
 · 
4 single beds
4 single beds
,
 · 
2 baths
2 baths
Jul 8 – 13
Jul 8 – 13
ج.م 132,014
ج.م 105,0...
  [2/5] Scraped: Superhost...
      Price: N/A | Rating: 4.78 | Reviews: 882
  [3/5] Scraped: Superhost...
      Price: N/A | Rating: N/A | Reviews: N/A
  [4/5] Scraped: N/A...
      Price: N/A |

In [12]:
print(len(scraped_data))

1
